# Collaboration and Competition — Shared DDPG

---

Train a **single shared DDPG agent** to control both Tennis players cooperatively.  
Both players share the same actor and critic networks. The environment is solved when the average score (max over both agents per episode) reaches **0.5** over 100 consecutive episodes.

### 1. Start the Environment

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import deque
from unityagents import UnityEnvironment
import torch

# Load the Tennis environment (Mac)
env = UnityEnvironment(file_name="Tennis.app")
brain_name = env.brain_names[0]
brain = env.brains[brain_name]

### 2. Explore the State and Action Spaces

In [ ]:
# Explore state and action spaces
env_info = env.reset(train_mode=True)[brain_name]
num_agents = len(env_info.agents)
action_size = brain.vector_action_space_size
states = env_info.vector_observations
state_size = states.shape[1]

print(f'Number of agents: {num_agents}')
print(f'State size (per agent): {state_size}')
print(f'Action size (per agent): {action_size}')
print(f'Sample state (agent 0): {states[0]}')

### 3. Create the Shared DDPG Agent

In [ ]:
from ddpg_agent import DDPGAgent

agent = DDPGAgent(state_size=state_size, action_size=action_size, seed=42)
print(f"Agent device: {next(agent.actor_local.parameters()).device}")
print(f"Actor params: {sum(p.numel() for p in agent.actor_local.parameters()):,}")
print(f"Critic params: {sum(p.numel() for p in agent.critic_local.parameters()):,}")

### 4. Define the Training Function

A single shared DDPG agent controls both Tennis players. Both agents' experiences are stored in the same replay buffer.

In [ ]:
def train_shared_ddpg(agent, n_episodes=5000, max_t=1000):
    """Train the shared DDPG agent in the Tennis environment."""
    scores_all = []
    scores_window = deque(maxlen=100)
    best_avg_score = -np.inf
    solved = False
    noise_scale = 1.0

    for i_episode in range(1, n_episodes + 1):
        env_info = env.reset(train_mode=True)[brain_name]
        states = env_info.vector_observations  # (2, 24)
        agent.reset()
        scores = np.zeros(num_agents)

        for t in range(max_t):
            actions = np.array([agent.act(states[i], noise_scale=noise_scale)
                                for i in range(num_agents)])

            env_info = env.step(actions)[brain_name]
            next_states = env_info.vector_observations
            rewards = env_info.rewards
            dones = env_info.local_done

            for i in range(num_agents):
                agent.step(states[i], actions[i], rewards[i], next_states[i], float(dones[i]))

            states = next_states
            scores += rewards

            if any(dones):
                break

        episode_score = np.max(scores)
        scores_all.append(episode_score)
        scores_window.append(episode_score)
        avg_score = np.mean(scores_window)

        noise_scale = max(0.1, noise_scale * 0.9999)

        print(f'\rEpisode {i_episode}\tAvg Score: {avg_score:.4f}\tNoise: {noise_scale:.4f}', end='')
        if i_episode % 100 == 0:
            print(f'\rEpisode {i_episode}\tAvg Score: {avg_score:.4f}\tNoise: {noise_scale:.4f}')

        if avg_score > best_avg_score:
            best_avg_score = avg_score
            agent.save()

        if avg_score >= 0.5 and not solved:
            print(f'\nSolved in {i_episode} episodes!\tAvg Score: {avg_score:.4f}')
            solved = True
            agent.save()
            break

    if not solved:
        print(f'\nTraining done. Best avg: {best_avg_score:.4f}')

    return scores_all

### 5. Run Training

In [ ]:
scores = train_shared_ddpg(agent, n_episodes=5000, max_t=1000)

### 6. Plot Scores

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
scores_arr = np.array(scores)
ax.plot(np.arange(1, len(scores_arr) + 1), scores_arr, alpha=0.3, label='Score')
if len(scores_arr) >= 100:
    moving_avg = np.convolve(scores_arr, np.ones(100) / 100, mode='valid')
    ax.plot(np.arange(100, len(scores_arr) + 1), moving_avg, label='100-episode avg')
ax.axhline(y=0.5, color='r', linestyle='--', label='Target (0.5)')
ax.set_xlabel('Episode')
ax.set_ylabel('Score')
ax.set_title('Shared DDPG Tennis Training')
ax.legend()
plt.show()
fig.savefig('scores_shared_plot.png', dpi=150, bbox_inches='tight')
print('Saved scores_shared_plot.png')

### 7. Watch a Trained Agent Play

In [ ]:
# Load trained weights and watch the agent play
trained_agent = DDPGAgent(state_size=state_size, action_size=action_size, seed=42)
trained_agent.load()

for i_episode in range(1, 4):
    env_info = env.reset(train_mode=False)[brain_name]
    states = env_info.vector_observations
    scores = np.zeros(num_agents)
    while True:
        actions = np.array([trained_agent.act(states[j], add_noise=False)
                            for j in range(num_agents)])
        env_info = env.step(actions)[brain_name]
        states = env_info.vector_observations
        rewards = env_info.rewards
        dones = env_info.local_done
        scores += rewards
        if any(dones):
            break
    print(f'Episode {i_episode}\tScore: {np.max(scores):.4f}')

env.close()